<img src="https://thumb.wikimedia.org/wikipedia/commons/thumb/2/2c/HEIG-VD_logotype-baseline_rouge-rvb.png/330px-HEIG-VD_logotype-baseline_rouge-rvb.png" alt="Logo HEIG-VD" style="width: 200px;" align="right"/>

# Cours APN - Labo 1 : Visualisation de vecteurs de mots

**Binôme :** Ismael Giovannetti, Mouhamed Sakho

## Résumé
Le but de ce laboratoire est de visualiser des vecteurs de mots grâce à des méthodes de réduction de dimensionnalité.  La visualisation permettra de deviner plus facilement le "mot du jour" dans le jeu en ligne [Cémantix](https://cemantix.certitudes.org/).

## 1. Prise en main de word2vec

Le modèle Word2vec contient des _embeddings_ des mots qui sont appris automatiquement à partir de textes.  Chaque mot est donc représenté par un vecteur dans un espace avec plusieurs centaines de dimensions (p.ex. 300 ou 500).  Dans cet espace, les mots de sens ou d'usage proches ont des vecteurs proches (les _embeddings_ capturent la similarité entre mots).  Mais la dimensionnalité de l'espace fait qu'il est difficile de visualiser les vecteurs de mots.

La librairie [Gensim](https://radimrehurek.com/gensim/index.html) permet de charger un modèle Word2vec existant (avec la fonction `load_word2vec_format(...)`) et de manipuler les vecteurs de mots (instances de la classe `KeyedVectors`).  On peut notamment utiliser la fonction `similarity(w1, w2)` qui retourne le cosinus des vecteurs correspondant aux mots `w1` et `w2`, à condition que ceux-ci soient connus du modèle.  On peut aussi utiliser la fonction `most_similar([w])` qui retourne les mots voisins d'un ou plusieurs mots.  La [documentation de KeyedVectors](https://radimrehurek.com/gensim/models/keyedvectors.html) fournit des exemples utiles.

Veuillez télécharger l'_un des deux_ modèles word2vec suivants, déjà entraînés, mis à disposition par [J.-Ph. Fauconnier](https://fauconnier.github.io/#data):
- [frWac_no_postag_no_phrase_500_cbow_cut100.bin](https://embeddings.net/embeddings/frWac_no_postag_no_phrase_500_cbow_cut100.bin) - plus petit (229 Mo), mais pas identique au modèle utilisé pour Cémantix ;
- [frWac_no_postag_phrase_500_cbow_cut10.bin](https://embeddings.net/embeddings/frWac_no_postag_phrase_500_cbow_cut10.bin) - bien plus grand (2 Go) donc assez gourmand en mémoire, mais qui est exactement celui utilisé pour Cémantix.
Veuillez placer le modèle choisi dans le dossier `gensim-data` de votre dossier utilisateur.

In [1]:
# Téléchargement, si nécessaire.
# import wget
# url = 'https://embeddings.net/embeddings/frWac_no_postag_phrase_500_cbow_cut10.bin'
# wget.download(url, path_to_model)


In [2]:
from gensim.models import KeyedVectors

In [3]:
import os
path_to_model = os.path.expanduser("~/gensim-data/frWac/frWac_no_postag_phrase_500_cbow_cut10.bin")


In [4]:
model = KeyedVectors.load_word2vec_format(path_to_model, binary=True, unicode_errors="ignore")

a. Choisissez deux mots de sens proches `w1` et `w2`, et un autre plus différent noté `w3`. Affichez la similarité selon word2vec (cosinus) entre chaque paire de mots.  Ces valeurs correspondent-elles à vos intuitions ?

In [5]:
# Deux mots proches (w1, w2) et un mot plus éloigné (w3)
w1, w2, w3 = 'chat', 'chien', 'ordinateur'

for a, b in [(w1, w2), (w1, w3), (w2, w3)]:
    print(f"similarité({a}, {b}) = {model.similarity(a, b):.3f}")

similarité(chat, chien) = 0.565
similarité(chat, ordinateur) = 0.134
similarité(chien, ordinateur) = 0.068


**Réponse 1a :** chat–chien = 0.565, chat–ordinateur = 0.134, chien–ordinateur = 0.068.

Cela correspond à l'intuition : les deux animaux domestiques sont nettement plus proches entre eux que de l'ordinateur, dont la similarité avec eux est presque nulle. On remarque toutefois que même deux mots très liés restent loin d'un cosinus de 1.

b. Affichez les 10 mots les plus proches de `w1` selon word2vec avec pour chacun sa similarité avec `w1`. 

In [6]:
# 10 plus proches voisins de w1, avec leur similarité cosinus
for word, sim in model.most_similar(w1, topn=10):
    print(f"{word:<20} {sim:.3f}")

minou                0.633
matou                0.627
petit_chatte         0.614
chatte               0.592
miauler              0.591
chaton               0.589
quoi_fouetter        0.583
félin                0.575
chien                0.565
chats                0.555


c. Quels sont les 15 premiers coefficients du vecteur (_embedding_) du mot `w1` ? Quelle est la dimension de ce vecteur ?  Quelle est la taille du vocabulaire connu du modèle ?  Vous pouvez simplement écrire les commandes répondant à ces questions.

In [7]:
print("15 premiers coefficients :", model[w1][:15])
print("Dimension des vecteurs   :", model.vector_size)
print("Taille du vocabulaire    :", len(model.key_to_index))

15 premiers coefficients : [-0.5510563  -0.22379288  0.72409904 -0.82713395 -0.68234926  2.0828469
 -0.30507645  0.400353    0.9901076  -1.2962811   1.2319442   1.5554144
  0.3478864  -1.6153904  -0.585831  ]
Dimension des vecteurs   : 500
Taille du vocabulaire    : 1081995


d. Veuillez écrire une fonction appelée `neighbors` selon les spécifications suivantes.  Cette fonction servira plus loin pour l'affichage.
* input : modèle, liste de mots, nombre de voisins (`topn`) ;
* output : liste de mots voisins de chacun des mots de l'input, représentés par un dictionnaire expliqué ci-après ;
* fonctionnement : pour _chacun_ des mots donnés en input, la fonction teste si le mot est dans le vocabulaire du modèle word2vec (sinon elle ne le considère pas), puis demande au modèle la liste des `topn` mots voisins ; pour chacun de ces mots, la fonction construit un `dict` à 4 champs : 
  - mot voisin
  - similarité
  - mot de départ
  - code de couleur associé au mot de départ : 1, 2, etc.

**Exemple** : si on appelle la fonction avec \['école'\], le début du résultat sera :
```
[{'neighbor': 'scolaire',
  'similarity': 0.7114928364753723,
  'ref_word': 'école',
  'color_code': 1}, ...
```

In [8]:
def neighbors(model, word_list, topn = 5):
    result = []
    color_code = 0
    for ref_word in word_list:
        # Les mots inconnus du modèle sont ignorés
        if ref_word not in model.key_to_index:
            print(f"'{ref_word}' absent du vocabulaire, ignoré.")
            continue
        color_code += 1
        for neighbor, similarity in model.most_similar(ref_word, topn=topn):
            result.append({'neighbor': neighbor,
                           'similarity': similarity,
                           'ref_word': ref_word,
                           'color_code': color_code})
    return result

In [9]:
# Tester avec cet appel :
print(neighbors(model, ['chat', 'étudier'], 3))

[{'neighbor': 'minou', 'similarity': 0.6334686875343323, 'ref_word': 'chat', 'color_code': 1}, {'neighbor': 'matou', 'similarity': 0.626587450504303, 'ref_word': 'chat', 'color_code': 1}, {'neighbor': 'petit_chatte', 'similarity': 0.6141418218612671, 'ref_word': 'chat', 'color_code': 1}, {'neighbor': 'étude', 'similarity': 0.6593688726425171, 'ref_word': 'étudier', 'color_code': 2}, {'neighbor': 'analyser', 'similarity': 0.6340749263763428, 'ref_word': 'étudier', 'color_code': 2}, {'neighbor': 'approfondir', 'similarity': 0.5609511733055115, 'ref_word': 'étudier', 'color_code': 2}]


## 2. Affichage simple des voisins d'un mot

Pour commencer, veuillez étudier et exécuter le code fourni en exemple : `Labo4_plotly.ipynb`.  Cela vous montrera comment utiliser l'affichage 2D/3D avec `plotly.js`, et vous permettra de traiter les questions suivantes.

Veuillez écrire et exécuter une fonction qui effectue les opérations suivantes, étant donné un mot en entrée :
* obtenir les vecteurs word2vec du mot et de ses `topn` plus proches voisins ;
* transformer ces vecteurs en 2D par l'ACP ;
* afficher en 2D chaque points correspondant à un mot, avec comme étiquette le mot lui-même.

**Notes:** cette fonction sert de version préliminaire à une fonction plus générique demandée ci-dessous.  Il n'est pas demandé de distinguer le mot entré des mots voisins, dans l'affichage (même type de points).  Il n'est pas nécessaire pour l'instant d'utiliser la fonction `neighbors`.

In [10]:
import plotly
plotly.io.renderers.default = 'iframe'
import numpy as np
import plotly.graph_objs as go
from sklearn.decomposition import PCA

In [11]:
def display_PCA_2D_neighbors(model, word, topn = 5):
    # Le mot lui-même suivi de ses topn voisins
    words = [word] + [w for w, _ in model.most_similar(word, topn=topn)]
    vectors = np.array([model[w] for w in words])

    # Projection des vecteurs en 2D
    points = PCA(n_components=2).fit_transform(vectors)

    plot_data = go.Scatter(x = points[:, 0], y = points[:, 1],
                           text = words,
                           textposition = "top center",
                           textfont_size = 10,
                           mode = 'markers+text',
                           marker = {'size': 8, 'opacity': 0.8})
    layout = go.Layout(margin = {'l': 0, 'r': 0, 'b': 0, 't': 0},
                       width = 800, height = 600,
                       font = {'family': 'Arial', 'size': 10})
    go.Figure(data = [plot_data], layout = layout).show()

In [12]:
# Afficher le mot 'chat' et ses 20 plus proches voisins
# projetés en deux dimensions avec PCA (test de la fonction).
display_PCA_2D_neighbors(model, 'chat', 20)

## 3. Affichage configurable des voisins de plusieurs mots

Le but maintenant est d'écrire une fonction `display_dimred_neighbors` qui affiche sur un seul graphique les mots voisins de chaque mot d'une liste de mots donnés.  Vous utiliserez ici la fonction auxiliaire `neighbors` définie plus haut.  Les paramètres suivants seront passés à la fonction `display_dimred_neighbors` :

* `model` - le nom du modèle word2vec
* `word_list` - liste de mots dont on veut afficher les voisins (s'ils existent dans `model`)
* `n_components` - dimensionnalité de l'affichage, 2 ou 3
* `topn` - nombre de voisins à afficher pour chaque mot 
* `method` - **méthode de réduction de dimensionnalité : pca, mds, isomap, tsne, ou umap**
* `n_neighbors`- nombre de voisins considérés par la méthode (applicable à isomap, umap, et tsne (appelé alors `perplexity`)).

Trois tests sont demandés à la question 4, mais vous pouvez en exécuter d'autres et les inclure ici.

In [13]:
from sklearn.decomposition import PCA
from sklearn.manifold import MDS
from sklearn.manifold import Isomap
from sklearn.manifold import TSNE
from umap import UMAP # Attention, installer le module nommé 'umap-learn' (avec conda install).

In [14]:
def display_dimred_neighbors(model, word_list, n_components = 3, topn = 5, method = 'pca', n_neighbors = 5):
    nb = neighbors(model, word_list, topn)
    if not nb:
        print("Aucun mot connu du modèle.")
        return

    # Points à afficher : les mots de départ, puis tous leurs voisins
    ref_colors = {d['ref_word']: d['color_code'] for d in nb}
    labels = list(ref_colors) + [d['neighbor'] for d in nb]
    groups = list(ref_colors) + [d['ref_word'] for d in nb]
    is_ref = [True] * len(ref_colors) + [False] * len(nb)
    vectors = np.array([model[w] for w in labels])

    # Réduction de dimensionnalité selon la méthode choisie
    reducers = {
        'pca':    lambda: PCA(n_components=n_components),
        'mds':    lambda: MDS(n_components=n_components, init='classical_mds', random_state=0),
        'isomap': lambda: Isomap(n_components=n_components, n_neighbors=n_neighbors),
        'tsne':   lambda: TSNE(n_components=n_components, perplexity=n_neighbors, random_state=0),
        'umap':   lambda: UMAP(n_components=n_components, n_neighbors=n_neighbors, random_state=0),
    }
    if method not in reducers:
        raise ValueError(f"Méthode inconnue : {method} (choisir parmi {list(reducers)})")
    points = reducers[method]().fit_transform(vectors)

    # Une trace par mot de départ : le mot en losange, ses voisins en ronds
    scatter = go.Scatter3d if n_components == 3 else go.Scatter
    palette = plotly.colors.qualitative.Plotly
    traces = []
    for ref_word, color_code in ref_colors.items():
        idx = [i for i, g in enumerate(groups) if g == ref_word]
        coords = {'x': points[idx, 0], 'y': points[idx, 1]}
        if n_components == 3:
            coords['z'] = points[idx, 2]
        traces.append(scatter(**coords,
                              text = [labels[i] for i in idx],
                              name = ref_word,
                              textposition = "top center",
                              textfont_size = 10,
                              mode = 'markers+text',
                              marker = {'size': [10 if is_ref[i] else 5 for i in idx],
                                        'symbol': ['diamond' if is_ref[i] else 'circle' for i in idx],
                                        'opacity': 0.8,
                                        'color': palette[(color_code - 1) % len(palette)]}))

    layout = go.Layout(title = f"{method.upper()} - topn={topn}, n_neighbors={n_neighbors}",
                       margin = {'l': 0, 'r': 0, 'b': 0, 't': 40},
                       width = 800, height = 600,
                       font = {'family': 'Arial', 'size': 10},
                       showlegend = True,
                       legend = {'x': 1, 'y': 0.5})
    go.Figure(data = traces, layout = layout).show()


## 4. Application à l'étude des mots voisins dans word2vec

Veuillez choisir quatre mots (`word_list = [m1, m2, m3, m4]`) de façon à ce que m1 et m2 soient très proches par leur sens ou leur usage, m3 un peu plus éloigné, et m4 très éloigné.  L'objectif de cette question est d'étudier la distribution des mots voisins de ces quatre mots (entre 10 et 50) par une visualisation en 2D ou en 3D des vecteurs de mots.  

En expérimentant avec plusieurs configurations (méthode de réduction de dimensionnalité, valeur de `n_neighbors`, nombre de voisins `topn`), veuillez répondre aux questions suivantes, à l'aide d'un ou plusieurs graphiques par question :

**a.** veuillez tester les deux méthodes de réduction de dimensionnalité linéaires (PCA et MDS métrique) et indiquer si les quatre groupes de mots sont clairement séparés dans word2vec.

**b.** veuillez tester les trois méthodes de réduction de dimensionnalité non-linéaires (Isomap, t-SNE et UMAP) et indiquer si on peut mieux mettre en évidence les quatre groupes de mots -- expérimentez avec divers paramètres et indiquez les meilleurs paramètres que vous avez trouvés, qui permettent de bien identifier les quatre ensembles.

**c.** veuillez chercher des paramètres qui aboutissent à cinq clusters ; même question avec trois clusters.

In [15]:
# Insérer ici la liste de mots.
# m1 et m2 très proches, m3 un peu plus éloigné, m4 très éloigné
word_list = ['voiture', 'bagnole', 'vélo', 'philosophie']

In [16]:
# Question 4a
# Méthodes linéaires : PCA puis MDS métrique
display_dimred_neighbors(model, word_list, n_components=2, topn=20, method='pca')
display_dimred_neighbors(model, word_list, n_components=2, topn=20, method='mds')

**Réponse 4a :** Le choix des mots respecte la consigne : voiture–bagnole = 0.62, voiture–vélo = 0.54, voiture–philosophie = −0.14.

- **PCA** : *philosophie* forme un groupe bien à part et *vélo* se détache assez nettement, mais les voisins de *voiture* et de *bagnole* sont mélangés. On voit donc trois groupes plutôt que quatre.
- **MDS métrique** : la séparation est encore moins bonne. Seul *philosophie* s'écarte, les voisins des trois mots liés au transport se superposent.

Les méthodes linéaires ne séparent donc pas clairement les quatre groupes. C'est attendu : *voiture* et *bagnole* ont des voisinages très proches (5 voisins en commun sur 20).

In [17]:
# Question 4b
# Meilleurs paramètres trouvés pour chaque méthode (après essais avec n_neighbors entre 3 et 50)
display_dimred_neighbors(model, word_list, n_components=2, topn=20, method='isomap', n_neighbors=10)
display_dimred_neighbors(model, word_list, n_components=2, topn=20, method='tsne', n_neighbors=8)
display_dimred_neighbors(model, word_list, n_components=2, topn=20, method='umap', n_neighbors=8)

p:\repos\HEIG-VD\APN_labs\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


**Réponse 4b :** Les méthodes non linéaires séparent mieux les groupes, à condition de garder un nombre de voisins assez faible.

- **Isomap** (`n_neighbors=10`) : quatre « branches » distinctes (philosophie, vélo, bagnole, voiture), mais voiture et bagnole restent en partie mélangés là où les branches se rejoignent.
- **t-SNE** (`perplexity=8`) : c'est le meilleur résultat, les quatre groupes sont identifiables (vélo, bagnole, voiture, et philosophie à l'écart). Dès 20, voiture et bagnole se mélangent.
- **UMAP** (`n_neighbors=8`) : philosophie est isolée et vélo se trouve à une extrémité de la bande des transports, mais voiture et bagnole y restent mélangés.

Meilleurs paramètres : **t-SNE avec une perplexité entre 8 et 12**. De manière générale, une petite valeur de `n_neighbors` (ou de perplexité) préserve la structure locale et sépare les groupes proches, tandis qu'une grande valeur privilégie la structure globale et fusionne voiture et bagnole.

On remarque aussi que les trois mots de départ *voiture*, *bagnole* et *vélo* (losanges) apparaissent souvent ensemble, un peu à l'écart de leurs voisins : chacun fait partie des voisins des autres.

In [18]:
# Question 4c
# Cinq clusters : un très petit n_neighbors fragmente les groupes
display_dimred_neighbors(model, word_list, n_components=2, topn=20, method='umap', n_neighbors=3)

# Trois clusters : un n_neighbors plus grand fusionne voiture et bagnole
display_dimred_neighbors(model, word_list, n_components=2, topn=20, method='umap', n_neighbors=15)

p:\repos\HEIG-VD\APN_labs\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


p:\repos\HEIG-VD\APN_labs\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


**Réponse 4c :**

- **Cinq clusters** avec UMAP et `n_neighbors=3` : philosophie, vélo, voiture+bagnole, un petit amas formé des mots de départ voiture/bagnole/vélo, et un petit amas secondaire de voisins de vélo. Un très petit nombre de voisins fragmente les groupes.
- **Trois clusters** avec UMAP et `n_neighbors=15` : philosophie, vélo, et un seul groupe voiture+bagnole. Un nombre de voisins plus grand fusionne les deux groupes les plus proches.

## 5. Outil d'assistance pour le jeu Cémantix

Le jeu en ligne [Cémantix](https://cemantix.certitudes.org/) (aussi proposé sur le site [Dictaly](https://www.dictaly.com/semantiques/))  demande de deviner le *mot du jour* en l'approchant peu à peu par des mots candidats.  Pour chacun, le système indique pour sa similarité word2vec avec le *mot du jour*, ce qui permet de se rapprocher graduellement de la solution.  Expérimentez d'abord un court instant avec le jeu 😀.

L'objectif de cette question est de visualiser les voisinages de mots, pour vous aider à proposer des mots candidats et trouver plus vite la solution.  La procédure est la suivante :
* essayez trois mots au hasard dans Cémantix
* affichez 20-30 mots voisins de ces trois mots grâce à la fonction `display_dimred_neighbors`
* à l'aide des mots affichés, essayez des mots candidats dans Cémantix
* changez l'affichage en remplaçant les trois mots par de meilleurs mots
* continuez jusqu'à trouver le *mot du jour*

**Notes**
* si vous le souhaitez, vous pouvez utiliser le modèle word2vec identique à celui de Cémantix, qui est frWac_no_postag_phrase_500_cbow_cut10.bin fourni par [J.-Ph. Fauconnier](https://fauconnier.github.io/#data)
* ce modèle fait environ 2.3 Go et contient aussi des expressions de plusieurs mots (séparés par '_')
* vous pouvez choisir d'utiliser le petit ou le grand modèle, sachant qu'avec le premier, les suggestions sont moins pertinentes
* Cémantix ignore les expressions de plusieurs mots présentes dans le grand modèle -- vous pouvez choisir de le faire ou non.

In [19]:
# Mots candidats retenus pendant la partie
candidate_words = ['période', 'phase', 'intervalle', 'processus', 'initial']
display_dimred_neighbors(model, candidate_words, n_components = 3, topn = 20, method = 'tsne')


Pour commencer, nous avons testé des mots de départ variés dans Cémantix (`animal`, `maison`, `travail`, `argent`, `temps`). Les mots `maison` et `argent` étaient très froids, mais `travail` (507‰) et `temps` (359‰) sont entrés dans le top 1000 et nous ont mis sur la piste.

En regardant les voisins autour de `travail` et `durée` avec `display_dimred_neighbors`, nous avons testé `période` qui est monté à 988‰. 

En affichant les voisins 3D de `période` (`phase`, `intervalle`, `processus`, `étape`), nous avons vu que tous ces mots tournaient autour de l'enchaînement temporel et des étapes d'un processus. Nous avons alors testé `initial` qui est monté à 999‰, puis nous avons trouvé le mot du jour : **`successif`** (1000‰) en 32 essais.

La visualisation 3D permet de voir rapidement des groupes de synonymes proches et d'explorer des pistes sémantiques cohérentes au lieu de chercher au hasard.

**Extrait de la partie :**
- Mot secret trouvé : **successif** (au 32e essai)
- Meilleurs mots de l'historique :
  - `successif` : 100 °C (1000‰) - Mot du jour
  - `initial` : 51.98 °C (999‰)
  - `période` : 44.97 °C (988‰)
  - `phase` : 41.35 °C (966‰)
  - `intervalle` : 41.07 °C (965‰)
  - `processus` : 40.31 °C (958‰)
  - `étape` : 39.15 °C (946‰)
  - `durant` : 34.16 °C (832‰)
  - `transition` : 34.14 °C (830‰)
  - `fin` : 34.06 °C (824‰)
  - `année` : 32.45 °C (747‰)
  - `laps` : 32.41 °C (743‰)
  - `début` : 30.43 °C (631‰)
  - `terme` : 30.32 °C (621‰)
  - `durée` : 29.43 °C (533‰)
  - `travail` : 29.21 °C (507‰)
  - `pendant` : 28.05 °C (371‰)
  - `temps` : 27.96 °C (359‰)


**Fin du Labo.**  Veuillez nettoyer ce notebook en gardant seulement les résultats désirés, l'enregistrer, et le soumettre comme devoir sur Cyberlearn.